<div align="center">

# **Tesis: Conectividad Estructural de Tractos Témporo-Límbicos en Epilepsia**
### Archivo 1 de 2 — Pipeline Tabular y de Clasificación Clínica (CLI Estadístico de DSI Studio)

</div>

Este notebook consume directamente las salidas nativas y ya validadas de **DSI Studio** (archivos
`*_tract_stat*.txt` generados con `dsi_studio --action=ana --export=stat`), evitando cualquier
decodificación manual del formato binario de tractografía. Esto elimina el riesgo de sesgos de
reconstrucción y garantiza la reproducibilidad exigida en una tesis de maestría: cualquier otro
investigador con acceso a DSI Studio y a los mismos datos crudos puede regenerar exactamente los
mismos archivos `.txt` que aquí se procesan.

**Complementa al Archivo 2** (`Vision_Computador_Clustering_Tractos.ipynb`), que sí trabaja a nivel
de streamlines individuales (`.tt.gz`) para validar geométricamente el emparejamiento
Izquierda-Derecha de los clústeres — pero como algoritmo de **verificación**, no como fuente de la
nomenclatura anatómica usada en el modelo estadístico final.

> ⚠️ **Corrección de alcance:** una versión previa de este pipeline estaba construida en torno a
> **esquizofrenia** (`grupo_dx: sz/control`) y a un circuito frontolímbico-lingüístico. El estudio
> real es sobre **epilepsia** (`grupo_dx: epilepsia/control`), con foco en cuatro tractos
> témporo-límbicos y de asociación temporoparietal (Sección 1.2). Todo el notebook —nomenclatura,
> patrones de nombre de archivo, métrica priorizada y referencias— se actualizó en consecuencia.


# **0. Configuración del entorno**
Detecta si el notebook corre en Google Colab (para habilitar subida de archivos) o en un entorno
local/Jupyter (donde se asume que los `.txt` ya están en `CARPETA_DATOS`).

In [ ]:
import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", font_scale=0.85)
pd.set_option("display.max_colwidth", 120)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CARPETA_SALIDA = "/content/" if IN_COLAB else "./"
os.makedirs(CARPETA_SALIDA, exist_ok=True)

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local / Jupyter'}")
print(f"Carpeta de salida: {CARPETA_SALIDA}")


In [ ]:
CARPETA_DATOS = CARPETA_SALIDA + "cohorte_stats/"
os.makedirs(CARPETA_DATOS, exist_ok=True)

if IN_COLAB:
    from google.colab import files
    print(f"Sube ahora todos los archivos '*_tract_stat*.txt' de la cohorte "
          f"(se guardarán en {CARPETA_DATOS}).")
    print("Puedes seleccionar varios archivos a la vez en el diálogo de subida.")
    subidos = files.upload()
    for nombre, contenido in subidos.items():
        with open(CARPETA_DATOS + nombre, "wb") as f:
            f.write(contenido)
else:
    # Ajusta esta ruta a la carpeta local (o de Drive montado) donde están los .txt de la cohorte
    CARPETA_DATOS = "/mnt/user-data/uploads/"

print(f"\nCarpeta de datos: {CARPETA_DATOS}")
print(f"Archivos '*tract_stat*.txt' encontrados: "
      f"{len(glob.glob(os.path.join(CARPETA_DATOS, '*tract_stat*.txt')))}")


# **1. Business Understanding (Entendimiento del Negocio)**

## **1.1. Objetivo del análisis**
Caracterizar la conectividad estructural de un conjunto de fascículos témporo-límbicos y de
asociación temporoparietal en pacientes con **epilepsia** (`epilepsia`) frente a controles sanos
(`control`), a partir de métricas escalares (FA, MD, AD, RD) y geométricas (volumen, longitud, span,
curl, elongación). Estos tractos se eligieron por su relación anatómica directa con el lóbulo
temporal y estructuras límbicas, la región más frecuentemente implicada como foco epileptógeno en
epilepsia focal (Concha, Beaulieu & Gross, 2005).

## **1.2. Fascículos de interés y su base anatómica**
La nomenclatura usada en todo este notebook es la de la **Terminologia Neuroanatomica (FIPAT, 2017)**
y de los estudios de disección virtual/tractografía originales de cada tracto, no etiquetas
heurísticas inventadas a partir de la posición geométrica:

| Código | Nombre científico | Rol funcional / relevancia en epilepsia |
|---|---|---|
| `arc` | Fascículo Arcuato | Vía dorsal del lenguaje (conexión perisilviana fronto-temporo-parietal); praxis. Su segmento posterior conecta con la región temporal a menudo afectada en epilepsia del lóbulo temporal (ELT) |
| `unc` | Fascículo Uncinado | Vía ventral fronto-límbica clásica (temporal anterior, incluida la amígdala, ↔ orbitofrontal); el tracto témporo-límbico más estudiado en ELT (Concha et al., 2005) |
| `mdlf` | Fascículo Longitudinal Medio (MdLF) | Conecta el polo temporal y el giro temporal superior con el lóbulo parietal (giro angular); lenguaje, atención, asociación auditiva de alto orden (Makris et al., 2009) |
| `ilf` | Fascículo Longitudinal Inferior | Conecta el lóbulo occipital con el polo y la cara medial del lóbulo temporal (incluida la amígdala/hipocampo); memoria visual, reconocimiento facial (Catani et al., 2003) |

> ⚠️ **Nota metodológica:** en versiones previas de este pipeline se usaron las etiquetas no
> estándar *"Fascículo Frontolímbico"* y *"Fascículo Frontoinsular"*, obtenidas por un heurístico
> geométrico (medialidad/altura) sin respaldo directo en la nomenclatura anatómica oficial, y el
> estudio estaba enmarcado en esquizofrenia en lugar de epilepsia. Este notebook descarta esas
> etiquetas (ver Sección 3.2) y usa en su lugar los nombres de tracto que el propio DSI Studio
> asigna durante el *Autotrack* (Yeh, 2022), presentes en el nombre de archivo de cada
> `*_tract_stat*.txt`.

## **1.3. Metodología: CRISP-DM**
Se sigue **CRISP-DM** (Cross Industry Standard Process for Data Mining): Entendimiento del Negocio →
Entendimiento de los Datos → Preparación de Datos → Modelado → Evaluación → Despliegue.

## **1.4. Referencias metodológicas clave**
- Concha, L., Beaulieu, C., Gross, D.W. "Bilateral limbic diffusion abnormalities in unilateral
  temporal lobe epilepsy." *Annals of Neurology* 57: 188-196, 2005 — motiva la elección de tractos
  témporo-límbicos como objeto de estudio en epilepsia (Sección 1.1).
- Catani, M., Jones, D.K., Donato, R., ffytche, D.H. "Occipito-temporal connections in the human
  brain." *Brain* 126: 2093-2107, 2003 — disección y nomenclatura del fascículo longitudinal
  inferior (ILF).
- Makris, N., Papadimitriou, G.M., Kaiser, J.R., Sorg, S., Kennedy, D.N., Pandya, D.N.
  "Delineation of the middle longitudinal fascicle in humans: a quantitative, in vivo, DT-MRI
  study." *Cerebral Cortex* 19(4): 777-785, 2009. DOI: 10.1093/cercor/bhn124 — disección y
  nomenclatura del fascículo longitudinal medio (MdLF).
- Catani, M., Thiebaut de Schotten, M. "A diffusion tensor imaging tractography atlas for virtual
  in vivo dissections." *Cortex* 44(8): 1105-1132, 2008. DOI: 10.1016/j.cortex.2008.05.004 —
  atlas de referencia para el fascículo arcuato y el fascículo uncinado.
- Yeh, F-C. "Population-based tract-to-region connectome of the human brain and its hierarchical
  topology." *Nature Communications* 13(1), 4933, 2022. DOI: 10.1038/s41467-022-32595-4
  (Autotrack, atlas ICBM152, espacio MNI).
- Yeh, F-C. et al. "Automatic Removal of False Connections in Diffusion MRI Tractography Using
  Topology-Informed Pruning (TIP)." *Neurotherapeutics*, 2019 — control de calidad post-tracking
  aplicado por DSI Studio antes de exportar las estadísticas que este notebook consume.
- Wakana, S. et al. "Reproducibility of quantitative tractography methods applied to cerebral white
  matter." *NeuroImage* 36(3): 630-644, 2007. DOI: 10.1016/j.neuroimage.2007.02.049.


# **2. Data Understanding (Entendimiento de los Datos)**

## **2.1. Formato de los archivos `*_tract_stat*.txt`**
Cada archivo es la salida de `dsi_studio --action=ana --source=<sujeto>.fz --tract=<tracto>.tt.gz
--export=stat` para **un tracto de un sujeto**: dos columnas separadas por tabulador, sin encabezado
(`métrica<TAB>valor`), por ejemplo:

```
number of tracts	118
number of seeds	91234
tract length mean(mm)	84.7
tract length sd(mm)	11.2
span mean(mm)	69.4
curl mean	1.18
elongation mean	3.41
diameter mean(mm)	4.62
volume(voxels)	14832
volume(mm3)	1498.6
fa mean	0.452
fa sd	0.081
md mean	0.00081
md sd	0.00012
ad mean	0.00112
ad sd	0.00015
rd mean	0.00064
rd sd	0.00010
```

Si tu instalación de DSI Studio exporta métricas adicionales o con otro nombre exacto, la función
`parsear_stat_txt` de la Sección 2.2 las conserva igualmente (normaliza el nombre pero no descarta
ninguna fila), así que no es necesario editar nada a mano.

## **2.2. Convención de nombres de archivo esperada**
Este notebook asume que el nombre de cada archivo contiene, en algún orden, el identificador del
sujeto, el grupo diagnóstico y el código del tracto con su lado, por ejemplo:

```
S1_nii_epi_corrected_tract_stat_arc_l.txt
S1_nii_epi_corrected_tract_stat_arc_r.txt
S2_nii_control_corrected_tract_stat_unc_l.txt
```

Si tu convención de nombres difiere, ajusta únicamente las expresiones regulares en
`PATRONES_NOMBRE` (Sección 2.3, primera celda) — el resto del pipeline no depende de esa convención.


In [ ]:
# ==========================================================================
# 2.3. Patrones de extracción a partir del nombre de archivo
# ==========================================================================
# Ajusta estos patrones si tu convención de nombres es distinta. Cada patrón
# se aplica sobre el nombre de archivo en minúsculas.

PATRONES_NOMBRE = {
    # subject_id: primer token que matchea "s" + dígitos
    "subject_id": re.compile(r"(s\d+)"),
    # grupo diagnóstico
    "grupo_epilepsia": re.compile(r"(?:_epi_|epi_corrected|_epilepsy_|_tle_)"),
    "grupo_control": re.compile(r"(?:_control_|_hc_|control_corrected)"),
    # lado
    "lado_izq": re.compile(r"(?:_l\b|_left|_izq)"),
    "lado_der": re.compile(r"(?:_r\b|_right|_der)"),
}

# Prefijos de tracto reconocidos -> nombre científico estándar (ver Sección 1.2).
# La búsqueda es por sub-cadena, insensible a mayúsculas, en el orden dado.
# Limitado a los 4 tractos témporo-límbicos de interés en epilepsia: si tu
# cohorte incluye otros tractos del Autotrack, agrégalos aquí con el mismo
# criterio (código de archivo -> nombre científico con fuente citable).
MAPA_TRACTOS = {
    "arc":  "Fascículo Arcuato",
    "unc":  "Fascículo Uncinado",
    "mdlf": "Fascículo Longitudinal Medio (MdLF)",
    "ilf":  "Fascículo Longitudinal Inferior (ILF)",
}


def extraer_metadatos_nombre(nombre_archivo):
    """Extrae subject_id, grupo_dx, tracto (código y nombre científico) y lado
    a partir del nombre de archivo, usando PATRONES_NOMBRE y MAPA_TRACTOS.
    Devuelve un dict; los campos que no se puedan inferir quedan como
    "desconocido" para que el archivo se procese igual y quede marcado
    para revisión manual en la bitácora (Sección 3.4).
    """
    n = nombre_archivo.lower()

    m_id = PATRONES_NOMBRE["subject_id"].search(n)
    subject_id = m_id.group(1).upper() if m_id else "desconocido"

    if PATRONES_NOMBRE["grupo_epilepsia"].search(n):
        grupo_dx = "epilepsia"
    elif PATRONES_NOMBRE["grupo_control"].search(n):
        grupo_dx = "control"
    else:
        grupo_dx = "desconocido"

    if PATRONES_NOMBRE["lado_izq"].search(n):
        lado = "Izquierdo"
    elif PATRONES_NOMBRE["lado_der"].search(n):
        lado = "Derecho"
    else:
        lado = "desconocido"

    tracto_codigo, tracto_nombre = "desconocido", "desconocido"
    for codigo, nombre_cientifico in MAPA_TRACTOS.items():
        if codigo in n:
            tracto_codigo, tracto_nombre = codigo, nombre_cientifico
            break

    return {
        "subject_id": subject_id,
        "grupo_dx": grupo_dx,
        "tracto_codigo": tracto_codigo,
        "tracto_nombre": tracto_nombre,
        "lado": lado,
    }


print("Funciones de extracción de metadatos listas.")
print("Prueba:", extraer_metadatos_nombre("S1_nii_epi_corrected_tract_stat_arc_l.txt"))


In [ ]:
# ==========================================================================
# 2.4. Parseo de un archivo *_tract_stat*.txt individual
# ==========================================================================
def normalizar_nombre_metrica(texto):
    """'tract length mean(mm)' -> 'tract_length_mean_mm'."""
    texto = texto.strip().lower()
    texto = re.sub(r"\(([^)]*)\)", r"_\1", texto)   # "(mm)" -> "_mm"
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    texto = re.sub(r"_+", "_", texto).strip("_")
    return texto


def parsear_stat_txt(ruta_archivo):
    """Lee un archivo DSI Studio '*.stat.txt' (2 columnas: métrica, valor,
    separadas por tabulador, sin encabezado) y lo devuelve como un dict
    {métrica_normalizada: valor_float}. Filas cuyo valor no sea numérico
    se conservan como texto (por robustez ante formatos ligeramente
    distintos entre versiones de DSI Studio).
    """
    tabla = pd.read_csv(ruta_archivo, sep="\t", header=None,
                         names=["metrica", "valor"], engine="python")
    fila = {}
    for _, r in tabla.iterrows():
        clave = normalizar_nombre_metrica(str(r["metrica"]))
        valor_bruto = r["valor"]
        try:
            valor = float(valor_bruto)
        except (TypeError, ValueError):
            valor = valor_bruto
        fila[clave] = valor
    return fila


print("Función 'parsear_stat_txt' lista.")


## **2.5. Inspección de un archivo de ejemplo**
Antes de consolidar toda la cohorte, se inspecciona un solo archivo para confirmar que el parseo
es correcto.

In [ ]:
archivos_stat = sorted(glob.glob(os.path.join(CARPETA_DATOS, "*tract_stat*.txt")))
print(f"Archivos de estadísticas por tracto encontrados: {len(archivos_stat)}")

if archivos_stat:
    ejemplo = archivos_stat[0]
    print(f"\nEjemplo: {os.path.basename(ejemplo)}")
    print("Metadatos inferidos del nombre:", extraer_metadatos_nombre(os.path.basename(ejemplo)))
    print("\nMétricas parseadas:")
    for k, v in parsear_stat_txt(ejemplo).items():
        print(f"  {k:30s} = {v}")
else:
    print("\nNo se encontraron archivos todavía. Sube o coloca los '*_tract_stat*.txt' "
          "en CARPETA_DATOS y vuelve a ejecutar esta celda.")


# **3. Preparación de Datos (CRISP-DM: Data Preparation)**

## **3.1. Consolidación: de N archivos individuales a un DataFrame maestro**
Se itera sobre todos los `*_tract_stat*.txt` de `CARPETA_DATOS`, se parsea cada uno (Sección 2.4) y
se combina con sus metadatos de nombre de archivo (Sección 2.3) en una sola fila. El resultado es un
DataFrame en **formato largo**: una fila por combinación (`subject_id`, `tracto`, `lado`).

## **3.2. Corrección de nomenclatura (regla crítica)**
`extraer_metadatos_nombre` ya asigna el nombre científico estándar (`tracto_nombre`) a partir del
código reconocido en el nombre de archivo (`MAPA_TRACTOS`, Sección 2.3) — por construcción, nunca
puede producir *"Frontolímbico"* ni *"Frontoinsular"*. Cualquier archivo cuyo código de tracto no
esté en `MAPA_TRACTOS` queda marcado como `"desconocido"` y se reporta explícitamente en la bitácora
de consolidación, en vez de recibir una etiqueta inventada.


In [ ]:
# ==========================================================================
# 3.3. Función de consolidación de la cohorte
# ==========================================================================
def consolidar_cohorte(carpeta):
    """Recorre todos los '*_tract_stat*.txt' de 'carpeta', los parsea y arma
    el DataFrame maestro en formato largo. Devuelve (df_maestro, df_bitacora).
    """
    filas = []
    bitacora = []

    for ruta in sorted(glob.glob(os.path.join(carpeta, "*tract_stat*.txt"))):
        nombre = os.path.basename(ruta)
        meta = extraer_metadatos_nombre(nombre)

        estado = "ok"
        advertencia = ""
        if "desconocido" in (meta["subject_id"], meta["grupo_dx"],
                              meta["tracto_codigo"], meta["lado"]):
            estado = "revisar_manual"
            advertencia = "No se pudo inferir subject_id/grupo_dx/tracto/lado a partir del nombre."

        try:
            metricas = parsear_stat_txt(ruta)
        except Exception as e:
            bitacora.append({"archivo": nombre, "estado": "fallo",
                              "advertencia": f"Error de parseo: {e}"})
            continue

        fila = {
            "subject_id": meta["subject_id"],
            "grupo_dx": meta["grupo_dx"],
            "tracto": meta["tracto_codigo"],
            "tracto_nombre": meta["tracto_nombre"],
            "lado": meta["lado"],
            "archivo_origen": nombre,
        }
        fila.update(metricas)
        filas.append(fila)

        bitacora.append({"archivo": nombre, "estado": estado, "advertencia": advertencia})

    df_maestro = pd.DataFrame(filas)
    df_bitacora = pd.DataFrame(bitacora)
    return df_maestro, df_bitacora


df_maestro, df_bitacora = consolidar_cohorte(CARPETA_DATOS)

print(f"Filas consolidadas: {len(df_maestro)}")
print(f"Sujetos únicos: {df_maestro['subject_id'].nunique() if len(df_maestro) else 0}")
if len(df_bitacora):
    n_revisar = (df_bitacora["estado"] != "ok").sum()
    print(f"Archivos que requieren revisión manual: {n_revisar} / {len(df_bitacora)}")

df_maestro.head(10)


In [ ]:
# ==========================================================================
# 3.4. Bitácora de consolidación — filas que no quedaron 100% identificadas
# ==========================================================================
df_bitacora[df_bitacora["estado"] != "ok"] if len(df_bitacora) else df_bitacora


## **3.5. Limpieza: columnas duplicadas de volumen y tipos de dato**
DSI Studio suele exportar el volumen tanto en vóxeles como en mm³; se conserva `volume_mm3` como
métrica geométrica principal. Se fuerzan a tipo numérico todas las columnas de métricas (las que no
son identificadoras), dejando `NaN` donde el valor no era numérico — nunca se inventan valores.

In [ ]:
COLUMNAS_ID = ["subject_id", "grupo_dx", "tracto", "tracto_nombre", "lado", "archivo_origen"]
columnas_metricas = [c for c in df_maestro.columns if c not in COLUMNAS_ID]

for c in columnas_metricas:
    df_maestro[c] = pd.to_numeric(df_maestro[c], errors="coerce")

n_faltantes = df_maestro[columnas_metricas].isna().sum()
if n_faltantes.sum() > 0:
    print("Valores no numéricos convertidos a NaN por columna:")
    print(n_faltantes[n_faltantes > 0])
else:
    print("Todas las métricas se convirtieron a numérico sin pérdidas.")

df_maestro.to_csv(CARPETA_SALIDA + "dataset_tractos_cohorte.csv", index=False, encoding="utf-8-sig")
print(f"\nDataset maestro guardado en: {CARPETA_SALIDA}dataset_tractos_cohorte.csv")
df_maestro.describe(include="all").T.head(15)


# **4. Data Understanding — Dashboard Compacto (Single-View)**
Un único lienzo con FA media, MD media y volumen por tracto (comparando `sz` vs. `control`), más
una tabla-resumen renderizada como subplot — todo en una sola figura, sin necesidad de *scroll*.

In [ ]:
def dashboard_compacto(df, columnas_metricas):
    """Genera un dashboard de una sola vista: 3 gráficos de barras (FA, MD, volumen
    por tracto y grupo) + 1 tabla resumen, todos como subplots de una misma figura.
    """
    metricas_clave = [m for m in ["fa_mean", "md_mean", "volume_mm3"] if m in columnas_metricas]
    if not metricas_clave:
        print("No se encontraron las métricas esperadas (fa_mean, md_mean, volume_mm3) "
              "para graficar; ajusta 'metricas_clave' según tus columnas reales.")
        return

    fig = plt.figure(figsize=(14, 7))
    gs = fig.add_gridspec(2, len(metricas_clave), height_ratios=[2.2, 1])

    for j, metrica in enumerate(metricas_clave):
        ax = fig.add_subplot(gs[0, j])
        datos = df.dropna(subset=[metrica])
        if len(datos) and "grupo_dx" in datos.columns and datos["grupo_dx"].nunique() > 1:
            sns.barplot(data=datos, x="tracto", y=metrica, hue="grupo_dx",
                        ax=ax, errorbar="sd", palette="Set2")
            ax.legend(fontsize=7, title=None)
        elif len(datos):
            sns.barplot(data=datos, x="tracto", y=metrica, ax=ax,
                        errorbar="sd", color="#4C72B0")
        ax.set_title(metrica, fontsize=10, fontweight="bold")
        ax.set_xlabel("")
        ax.tick_params(axis="x", labelrotation=45, labelsize=7)
        ax.tick_params(axis="y", labelsize=7)

    ax_tabla = fig.add_subplot(gs[1, :])
    ax_tabla.axis("off")
    resumen = (
        df.groupby(["tracto", "grupo_dx"])[metricas_clave]
        .mean().round(3).reset_index()
        if "grupo_dx" in df.columns else
        df.groupby("tracto")[metricas_clave].mean().round(3).reset_index()
    )
    tabla = ax_tabla.table(
        cellText=resumen.values,
        colLabels=resumen.columns,
        loc="center",
        cellLoc="center",
    )
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(7)
    tabla.scale(1, 1.3)

    plt.suptitle("Dashboard compacto — Métricas por tracto y grupo diagnóstico",
                  fontsize=12, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    ruta_png = CARPETA_SALIDA + "dashboard_compacto.png"
    plt.savefig(ruta_png, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"Dashboard guardado en: {ruta_png}")


if len(df_maestro):
    dashboard_compacto(df_maestro, columnas_metricas)
else:
    print("df_maestro está vacío; sube datos y vuelve a ejecutar desde la Sección 2.")


# **5. Modelado (CRISP-DM: Modeling)**

## **5.1. Por qué `GroupShuffleSplit`**
Cada sujeto aporta **8 filas** (4 tractos × 2 lados) al dataset en formato largo. Si la partición
train/test se hiciera fila por fila, streamlines del mismo sujeto podrían aparecer en ambos
conjuntos — **fuga de datos** (*data leakage*) que infla artificialmente el desempeño. Por eso se
usa `GroupShuffleSplit` agrupado por `subject_id`: todas las filas de un mismo sujeto quedan del
mismo lado de la partición.

## **5.2. Preparación de la matriz de features**
Se pivota el DataFrame largo a **una fila por sujeto**, con columnas
`<tracto>_<lado>_<métrica>` (p. ej. `arc_izquierdo_fa_mean`), para que cada observación del modelo
sea un paciente completo, no un tracto aislado.

In [ ]:
def construir_matriz_features(df, columnas_metricas):
    """Pivota el DataFrame largo (una fila por tracto-lado-sujeto) a una matriz
    ancha con una fila por sujeto y columnas '<tracto>_<lado>_<métrica>'.
    """
    df_ancho = df.pivot_table(
        index=["subject_id", "grupo_dx"],
        columns=["tracto", "lado"],
        values=columnas_metricas,
        aggfunc="first",
    )
    df_ancho.columns = [f"{tracto}_{lado}_{metrica}".lower()
                         for metrica, tracto, lado in df_ancho.columns]
    df_ancho = df_ancho.reset_index()
    return df_ancho


if len(df_maestro):
    df_features = construir_matriz_features(df_maestro, columnas_metricas)
    print(f"Sujetos: {len(df_features)}   Features por sujeto: "
          f"{df_features.shape[1] - 2}")
    df_features.to_csv(CARPETA_SALIDA + "features_por_sujeto.csv", index=False,
                        encoding="utf-8-sig")
    df_features.head()


In [ ]:
from sklearn.model_selection import GroupShuffleSplit, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import make_scorer, recall_score, roc_auc_score, classification_report

MODELOS_CANDIDATOS = {
    "Regresión Logística": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight="balanced"),
    "SVM": SVC(probability=True, random_state=42, class_weight="balanced"),
    "KNN": KNeighborsClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

SCORERS = {
    "recall": make_scorer(recall_score, pos_label="epilepsia", zero_division=0),
    "roc_auc": "roc_auc",
}

print("Modelos candidatos:", list(MODELOS_CANDIDATOS.keys()))
print("Métricas priorizadas: Recall (clase 'epilepsia') y AUC-ROC.")


## **5.3. Pipeline de entrenamiento y validación agrupada**
`entrenar_y_evaluar` recibe la matriz de features por sujeto y ejecuta, para cada uno de los 5
clasificadores: imputación de faltantes → estandarización (`StandardScaler`, ajustado **solo** con
el fold de entrenamiento, dentro del pipeline, para no filtrar información) → validación cruzada
agrupada por `subject_id` (`GroupShuffleSplit`) → cálculo de Recall y AUC-ROC.

Este bloque se activa automáticamente en cuanto `df_features` contenga sujetos de **ambas clases**
de `grupo_dx` (`epilepsia` y `control`); con una sola clase, `roc_auc` no está definido y el bloque
lo informa en vez de fallar.

In [ ]:
def entrenar_y_evaluar(df_features, columnas_excluir=("subject_id", "grupo_dx"),
                        n_splits=5, test_size=0.25, random_state=42):
    """Ejecuta validación cruzada agrupada por subject_id para cada modelo de
    MODELOS_CANDIDATOS y devuelve un DataFrame con Recall y AUC-ROC (media y
    desviación estándar) por modelo.
    """
    y = df_features["grupo_dx"]
    if y.nunique() < 2:
        print("Se requieren al menos 2 clases distintas en 'grupo_dx' (p. ej. 'epilepsia' y "
              "'control') para entrenar y evaluar. Actualmente hay:", y.unique().tolist())
        return None

    X = df_features.drop(columns=list(columnas_excluir))
    grupos = df_features["subject_id"]

    splitter = GroupShuffleSplit(n_splits=n_splits, test_size=test_size,
                                  random_state=random_state)

    filas_resultado = []
    for nombre_modelo, estimador in MODELOS_CANDIDATOS.items():
        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clasificador", estimador),
        ])
        resultados = cross_validate(
            pipeline, X, y, groups=grupos, cv=splitter,
            scoring=SCORERS, error_score="raise",
        )
        filas_resultado.append({
            "modelo": nombre_modelo,
            "recall_media": resultados["test_recall"].mean(),
            "recall_sd": resultados["test_recall"].std(),
            "auc_roc_media": resultados["test_roc_auc"].mean(),
            "auc_roc_sd": resultados["test_roc_auc"].std(),
        })

    return pd.DataFrame(filas_resultado).sort_values("auc_roc_media", ascending=False)


if len(df_maestro) and df_maestro["grupo_dx"].nunique() > 1:
    df_resultados_modelos = entrenar_y_evaluar(df_features)
    if df_resultados_modelos is not None:
        df_resultados_modelos.to_csv(
            CARPETA_SALIDA + "resultados_modelado.csv", index=False, encoding="utf-8-sig"
        )
    df_resultados_modelos
else:
    print("El dataset actual no tiene ambas clases ('epilepsia' y 'control') representadas "
          "todavía; este bloque se activará automáticamente en cuanto la cohorte esté completa.")


# **6. Evaluación (CRISP-DM: Evaluation)**

## **6.1. Por qué Recall es la métrica prioritaria**
En un contexto clínico de cribado de epilepsia, un **falso negativo** (paciente con `epilepsia`
clasificado como `control`) es más costoso que un falso positivo: implica no derivar a un paciente
que sí necesita seguimiento y manejo antiepiléptico. Por eso se reporta y se prioriza el **Recall de
la clase `epilepsia`** por encima del *accuracy* global, complementado con **AUC-ROC** para evaluar
la separabilidad de las clases independientemente del umbral de decisión elegido.

## **6.2. Reporte de clasificación detallado del mejor modelo**
Entrena el modelo con mejor AUC-ROC promedio (Sección 5.3) sobre una única partición agrupada, y
muestra el reporte de clasificación completo (precisión, recall y F1 por clase) para inspección
cualitativa — complementario a las métricas promediadas por validación cruzada.

In [ ]:
def reporte_mejor_modelo(df_features, df_resultados_modelos,
                          columnas_excluir=("subject_id", "grupo_dx"),
                          test_size=0.25, random_state=42):
    mejor_nombre = df_resultados_modelos.iloc[0]["modelo"]
    print(f"Mejor modelo por AUC-ROC promedio: {mejor_nombre}")

    y = df_features["grupo_dx"]
    X = df_features.drop(columns=list(columnas_excluir))
    grupos = df_features["subject_id"]

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    idx_train, idx_test = next(splitter.split(X, y, groups=grupos))

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clasificador", MODELOS_CANDIDATOS[mejor_nombre]),
    ])
    pipeline.fit(X.iloc[idx_train], y.iloc[idx_train])
    y_pred = pipeline.predict(X.iloc[idx_test])

    print("\nReporte de clasificación (partición de prueba agrupada por sujeto):")
    print(classification_report(y.iloc[idx_test], y_pred, zero_division=0))
    return pipeline


if len(df_maestro) and df_maestro["grupo_dx"].nunique() > 1 and df_resultados_modelos is not None:
    pipeline_final = reporte_mejor_modelo(df_features, df_resultados_modelos)
else:
    print("Se ejecutará en cuanto haya resultados de modelado disponibles (Sección 5.3).")


# **7. Despliegue (CRISP-DM: Deployment) — Persistencia de Artefactos**
Se guardan como archivos reutilizables por el equipo: el dataset maestro (formato largo), la matriz
de features por sujeto, los resultados comparativos de los 5 modelos y la bitácora de consolidación,
además de un archivo de referencias metodológicas trazando cada paso a su fuente original.

In [ ]:
referencias_md = """# Referencias Metodológicas — Pipeline Tabular / Clasificación (Archivo 1)

## Formato de datos y software
Sitio oficial DSI Studio: https://dsi-studio.labsolver.org/
Formatos de datos y CLI: https://dsi-studio.labsolver.org/doc/cli_data.html

Yeh, Fang-Cheng. "DSI Studio: An Integrated Tractography Platform and Fiber Data Hub for
Accelerating Brain Research." Nature Methods, July 2025. DOI: 10.1038/s41592-025-02762-8

## Anatomía y nomenclatura de tractos (Sección 1.2)
Catani, M., Thiebaut de Schotten, M. "A diffusion tensor imaging tractography atlas for virtual
in vivo dissections." Cortex 44(8): 1105-1132, 2008. DOI: 10.1016/j.cortex.2008.05.004

Catani, M., Jones, D.K., Donato, R., ffytche, D.H. "Occipito-temporal connections in the human
brain." Brain 126: 2093-2107, 2003. (Disección del fascículo longitudinal inferior, ILF)

Makris, N., Papadimitriou, G.M., Kaiser, J.R., Sorg, S., Kennedy, D.N., Pandya, D.N.
"Delineation of the middle longitudinal fascicle in humans: a quantitative, in vivo, DT-MRI
study." Cerebral Cortex 19(4): 777-785, 2009. DOI: 10.1093/cercor/bhn124 (fascículo longitudinal
medio, MdLF)

Wakana, S. et al. "Reproducibility of quantitative tractography methods applied to cerebral
white matter." NeuroImage 36(3): 630-644, 2007. DOI: 10.1016/j.neuroimage.2007.02.049

FIPAT. Terminologia Neuroanatomica. Federative International Programme for Anatomical
Terminology, 2017. FIPAT.library.dal.ca

## Relevancia clínica en epilepsia (Sección 1.1)
Concha, L., Beaulieu, C., Gross, D.W. "Bilateral limbic diffusion abnormalities in unilateral
temporal lobe epilepsy." Annals of Neurology 57: 188-196, 2005.

## Reconstrucción, atlas poblacional y control de calidad (Autotrack, TIP)
Yeh, F-C. et al. "Deterministic diffusion fiber tracking improved by quantitative anisotropy."
PLoS ONE 8(11): e80713, 2013.

Yeh, F-C. "Population-based tract-to-region connectome of the human brain and its hierarchical
topology." Nature Communications 13(1), 4933, 2022. DOI: 10.1038/s41467-022-32595-4

Yeh, F-C. et al. "Automatic Removal of False Connections in Diffusion MRI Tractography Using
Topology-Informed Pruning (TIP)." Neurotherapeutics, 2019.

## Anisotropía Fraccional (FA): interpretación (Sección 1, nota conceptual)
La FA es un índice inespecífico modulado por la anatomía axonal (diámetro, densidad), la densidad
de mielina, la distancia internodal y la dispersión geométrica de las fibras (cruce/ramificación)
— no un marcador directo de "pérdida de mielina" (Beaulieu, 2002; Song et al., 2003; Jones et al.,
2013, NeuroImage 73: 239-254, "White matter integrity, fiber count, and other fallacies").
"""

with open(CARPETA_SALIDA + "referencias_dsi_studio_archivo1.md", "w", encoding="utf-8") as f:
    f.write(referencias_md)

print(f"Artefactos guardados en {CARPETA_SALIDA}:")
print(" - dataset_tractos_cohorte.csv        (formato largo: 1 fila por sujeto-tracto-lado)")
print(" - features_por_sujeto.csv            (formato ancho: 1 fila por sujeto)")
print(" - resultados_modelado.csv            (si había ≥2 clases en grupo_dx)")
print(" - dashboard_compacto.png")
print(" - referencias_dsi_studio_archivo1.md")


# **8. Conclusión**
- Se implementó un pipeline **CRISP-DM completo**, reproducible y sin decodificación manual de
  binarios, que consume directamente las salidas estadísticas nativas de DSI Studio por tracto.
- Se corrigió tanto el **alcance clínico** (epilepsia, no esquizofrenia) como la **nomenclatura
  anatómica**: los 4 tractos témporo-límbicos de interés (Arcuato, Uncinado, MdLF, ILF) se
  identifican por el código que el propio Autotrack de DSI Studio asigna (Sección 1.2,
  `MAPA_TRACTOS`), citando explícitamente Catani et al. (2003, 2008), Makris et al. (2009), Wakana
  et al. (2007) y la Terminologia Neuroanatomica — sin usar las etiquetas heurísticas
  *"Frontolímbico"* / *"Frontoinsular"* de versiones anteriores.
- El pipeline de modelado (`GroupShuffleSplit` + `StandardScaler` + 5 clasificadores) queda
  **activo por diseño**: corre automáticamente en cuanto `CARPETA_DATOS` contenga sujetos
  `epilepsia` y `control`, sin necesidad de modificar código.
- **Siguiente paso recomendado:** ejecutar `dsi_studio --action=ana --export=stat` sobre cada uno
  de los 4 tractos (Arcuato, Uncinado, MdLF, ILF) de cada sujeto de la cohorte completa, depositar
  los `.txt` resultantes en `CARPETA_DATOS` con el patrón de nombre `..._epi_..._<código>_<lado>.txt`
  o `..._control_..._<código>_<lado>.txt`, y volver a ejecutar este notebook de principio a fin.
